In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# 1) Upload file
uploaded = files.upload()
file_path = r"C:\Users\HP\Documents\Combined stock analysis.xlsx"

# 2) Load sheets
crdb = pd.read_excel(file_path, sheet_name="CRDB")
tbl  = pd.read_excel(file_path, sheet_name="TBL")
tcc  = pd.read_excel(file_path, sheet_name="TCC")

crdb["Stock"] = "CRDB"; tbl["Stock"] = "TBL"; tcc["Stock"] = "TCC"
df = pd.concat([crdb, tbl, tcc], ignore_index=True)
df["Date"] = pd.to_datetime(df["Date"])

# 3) Align to daily index & returns
full_dates = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
aligned = []
for stock, g in df.groupby("Stock"):
    g = g.set_index("Date").sort_index().reindex(full_dates)
    g["Adj. Close"] = g["Adj. Close"].ffill()
    g["Stock"] = stock
    aligned.append(g.reset_index().rename(columns={"index":"Date"}))
df = pd.concat(aligned, ignore_index=True).sort_values(["Stock","Date"])
df["Daily_Return"] = df.groupby("Stock")["Adj. Close"].pct_change(fill_method=None)

pivot_returns = df.pivot(index="Date", columns="Stock", values="Daily_Return").dropna(how="any")

# 4) Preview
print("Daily Returns\n")
print(pivot_returns.head(5))
print("...\n...\n...")
print(pivot_returns.tail(20).to_string(header=False))

# 5) Summary statistics
def describe_series(x):
    q = x.quantile([0.25,0.5,0.75])
    return pd.Series({
        "Count": x.count(),
        "Min": x.min(),
        "Max": x.max(),
        "Mean": x.mean(),
        "Std": x.std(ddof=1),
        "Skewness": x.skew(),
        "Kurtosis": x.kurtosis(),
        "Q1": q.loc[0.25],
        "Median": q.loc[0.5],
        "Q3": q.loc[0.75]
    })
summary_stats = pivot_returns.apply(describe_series)
print("\nSummary statistics (daily returns)")
print(summary_stats)

# 6) Dependence
pearson_corr  = pivot_returns.corr(method="pearson")
spearman_corr = pivot_returns.corr(method="spearman")
kendall_corr  = pivot_returns.corr(method="kendall")
print("\nPearson correlation"); print(pearson_corr)
print("\nSpearman rank correlation"); print(spearman_corr)
print("\nKendall's tau"); print(kendall_corr)

# Heatmaps
plt.figure(figsize=(6,5)); sns.heatmap(pearson_corr, annot=True, cmap="coolwarm", center=0); plt.title("Pearson correlation"); plt.show()
plt.figure(figsize=(6,5)); sns.heatmap(spearman_corr, annot=True, cmap="coolwarm", center=0); plt.title("Spearman correlation"); plt.show()
plt.figure(figsize=(6,5)); sns.heatmap(kendall_corr, annot=True, cmap="coolwarm", center=0); plt.title("Kendall's tau"); plt.show()

# 7) Risk metrics
levels = [0.90, 0.95, 0.99, 0.995, 0.999]
z_map = {0.90:1.2816,0.95:1.6449,0.99:2.3263,0.995:2.5758,0.999:3.0902}

def hist_var(x, cl):
    return -x.quantile(1-cl)

def hist_es(x, cl):
    var = x.quantile(1-cl)
    tail = x[x <= var]
    return -tail.mean() if len(tail) else np.nan

def para_var(x, cl):
    mu, sigma = x.mean(), x.std(ddof=1); z = z_map[cl]
    return -(mu - z*sigma)

def para_es(x, cl):
    mu, sigma = x.mean(), x.std(ddof=1); z = z_map[cl]
    phi = np.exp(-0.5*z*z)/np.sqrt(2*np.pi)
    return -(mu + sigma*(phi/(1-cl)))

wealth, drawdowns = {}, {}
for col in pivot_returns.columns:
    r = pivot_returns[col].fillna(0.0)
    w = (1.0+r).cumprod(); peak = w.cummax()
    dd = (w/peak - 1.0)
    wealth[col], drawdowns[col] = w, dd
drawdowns_df = pd.DataFrame(drawdowns)

def max_drawdown(dd): return dd.min()
def cdar(dd, cl):
    thr = dd.quantile(1-cl)
    worst = dd[dd <= thr]
    return worst.mean()

hist_var_table = pd.DataFrame({f"VaR_hist_{int(cl*100)}%": pivot_returns.apply(lambda x: hist_var(x, cl)) for cl in levels})
hist_es_table  = pd.DataFrame({f"ES_hist_{int(cl*100)}%":  pivot_returns.apply(lambda x: hist_es(x, cl))  for cl in levels})
para_var_table = pd.DataFrame({f"VaR_param_{int(cl*100)}%": pivot_returns.apply(lambda x: para_var(x, cl)) for cl in levels})
para_es_table  = pd.DataFrame({f"ES_param_{int(cl*100)}%":  pivot_returns.apply(lambda x: para_es(x, cl))  for cl in levels})
mdd_table = pd.DataFrame({"Max_Drawdown": drawdowns_df.apply(max_drawdown)})
cdar_table = pd.DataFrame({f"CDaR_{int(cl*100)}%": drawdowns_df.apply(lambda s: cdar(s, cl)) for cl in levels})

print("\nHistorical VaR"); print(hist_var_table)
print("\nParametric VaR"); print(para_var_table)
print("\nHistorical ES"); print(hist_es_table)
print("\nParametric ES"); print(para_es_table)
print("\nMaximum drawdown"); print(mdd_table)
print("\nConditional drawdown at risk (CDaR)"); print(cdar_table)

rolling_vol_30 = pivot_returns.rolling(30).std(ddof=1)
print("\nRolling 30-day volatility (last 10 days)")
print(rolling_vol_30.tail(10).to_string(header=False))

# 8) Plots: Daily Returns
plt.figure(figsize=(12,6))
for col in pivot_returns.columns:
    plt.plot(pivot_returns.index, pivot_returns[col], label=col)
plt.title("Daily Returns of CRDB, TBL, and TCC")
plt.xlabel("Date"); plt.ylabel("Daily Return")
plt.legend(); plt.grid(True); plt.show()

fig, axes = plt.subplots(3,1,figsize=(12,10),sharex=True)
for (stock, ax) in zip(pivot_returns.columns, axes):
    ax.plot(pivot_returns.index, pivot_returns[stock], label=stock)
    ax.set_title(f"Daily Returns - {stock}")
    ax.set_ylabel("Return"); ax.legend(); ax.grid(True)
plt.xlabel("Date"); plt.tight_layout(); plt.show()

# 9) Plots: Skewness distributions
fig, axes = plt.subplots(3,1,figsize=(12,10))
for (stock, ax) in zip(pivot_returns.columns, axes):
    temp = pivot_returns[stock].dropna()
    sns.histplot(temp, bins=50, kde=True, ax=ax)
    ax.set_title(f"Distribution & Skewness - {stock}\nSkew={temp.skew():.4f}")
    ax.set_xlabel("Daily Return"); ax.set_ylabel("Frequency")
plt.tight_layout(); plt.show()

ModuleNotFoundError: No module named 'google.colab'